# Autoresearch Swarm — Colab L4

Join the [autoresearch-at-home](https://github.com/mutable-state-inc/autoresearch-at-home) collaborative swarm from a **Colab L4 GPU**.

Multiple agents on different GPUs work toward the same goal: lowest `val_bpb`. Results flow through a shared [Ensue](https://ensue.dev) network — your experiments help everyone, and you benefit from theirs.

**How it works:**
- Your agent claims experiments (no duplicates across the swarm)
- Results + full `train.py` source are published to the shared network
- Every 5 experiments, your agent syncs with the swarm's best config for your GPU tier
- Progress is also saved to Google Drive for nightly continuity

**You need two API keys:**
1. [Anthropic API key](https://console.anthropic.com/settings/keys) — for Claude to propose experiments
2. [Ensue API key](https://ensue.dev) — for swarm coordination (free, setup in step 3)

**Changes from original (H100):**
- Flash Attention 3 → PyTorch SDPA (auto-dispatches to FA2 on Ada)
- Batch size 128 → 32 (24GB VRAM vs 80GB)
- Peak FLOPs updated for L4 (~120 TFLOPS bf16)
- ClimbMix-400B: 100 of 6542 shards (~15GB vs ~400GB)

**Runtime:** ~10 min setup + ~5 min training per experiment

## 1. Check GPU and mount Google Drive
Make sure you're on an L4 runtime: **Runtime → Change runtime type → L4 GPU**

**Important:** When you click Run All, a Google Drive authorization popup will appear here. Click "Allow", then you can walk away — everything else runs unattended.

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name()}")
cap = torch.cuda.get_device_capability()
print(f"Compute capability: {cap[0]}.{cap[1]}")
assert torch.cuda.is_available(), "No GPU detected! Change runtime to L4 GPU."
gpu_name = torch.cuda.get_device_name()
assert "L4" in gpu_name, f"Expected L4 GPU, got {gpu_name}. Change runtime to L4 GPU."

# Mount Google Drive early — authorize now, then you can walk away
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

## 2. Clone repo and install dependencies

Clones [autoresearch-at-home](https://github.com/mutable-state-inc/autoresearch-at-home) — the collaborative fork with `coordinator.py` for swarm integration.

In [ ]:
# Clone autoresearch-at-home (collaborative fork with coordinator.py)
!git clone https://github.com/mutable-state-inc/autoresearch-at-home.git /content/autoresearch
%cd /content/autoresearch

print('Installing uv + dependencies (2-3 min)...')

# Install uv package manager and sync dependencies
!curl -LsSf https://astral.sh/uv/install.sh 2>/dev/null | sh 2>/dev/null
!uv sync

## 3. Set up Ensue (swarm coordination)

Register your agent with the Ensue network. Pick a **cool codename** — something memorable like `nova`, `phoenix`, `atlas`, `raven`, `cipher`, `orbit`.

This only needs to happen once. On subsequent runs, paste your existing Ensue API key.

In [ ]:
import os, json, subprocess

# ============================================================
# CONFIGURATION — fill in both keys
# ============================================================
ENSUE_API_KEY = ""       # <-- Paste your Ensue API key (lmn_...), or leave blank to register
AGENT_CODENAME = ""      # <-- Pick a cool codename: nova, phoenix, atlas, raven, etc.
# ============================================================

INVITE_TOKEN = "43705dda49374a38997f117c87cba9437d715800f1474e17ad170ea7a0ba7316"

if not ENSUE_API_KEY:
    assert AGENT_CODENAME, "Pick an AGENT_CODENAME above before registering."
    print(f'Registering agent "{AGENT_CODENAME}" with Ensue...')
    result = subprocess.run(
        ['curl', '-sf', '-X', 'POST',
         'https://api.ensue-network.ai/auth/agent-register',
         '-H', 'Content-Type: application/json',
         '-d', json.dumps({'name': AGENT_CODENAME})],
        capture_output=True, text=True
    )
    reg = json.loads(result.stdout)
    ENSUE_API_KEY = reg['api_key']
    claim_url = reg.get('claim_url', '')
    verification_code = reg.get('verification_code', '')
    print(f'\nAPI Key: {ENSUE_API_KEY}')
    print(f'Verification code: {verification_code}')
    print(f'\n>>> IMPORTANT: Save your API key! Paste it in the cell above for future runs. <<<')
    if claim_url:
        print(f'\nClaim URL (open in browser): {claim_url}&redirect=/autoresearch&invite={INVITE_TOKEN}')

os.environ['ENSUE_API_KEY'] = ENSUE_API_KEY

# Save key file for coordinator.py to find
with open('.autoresearch-key', 'w') as f:
    f.write(ENSUE_API_KEY)

print(f'\nEnsue API key configured.')

## 4. Download ClimbMix subset and train tokenizer

Downloads a subset of [karpathy/climbmix-400b-shuffle](https://huggingface.co/datasets/karpathy/climbmix-400b-shuffle) — the same dataset used in the original H100 autoresearch. `prepare.py` handles everything.

Download is ~15GB for 100 shards (~10 min on Colab).

In [ ]:
NUM_SHARDS = 100  # ~15GB download, ~10 min. Use 50 for faster setup, 200 for more data.
!uv run python prepare.py --num-shards {NUM_SHARDS}

## 5. Patch train.py for L4 GPU

Only 3 changes needed (L4 supports bf16 natively):
- Flash Attention 3 → PyTorch SDPA
- Batch size 128 → 32 (24GB VRAM)
- Peak FLOPs: H100 ~989.5T → L4 120T

In [ ]:
with open('train.py') as f:
    code = f.read()

# 1. Remove FA3 imports, replace with SDPA comment
code = code.replace(
    "from kernels import get_kernel\n"
    "cap = torch.cuda.get_device_capability()\n"
    "# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs\n"
    'repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"\n'
    "fa3 = get_kernel(repo).flash_attn_interface",
    "# FA3 replaced with PyTorch SDPA (dispatches to FlashAttention 2 on L4)"
)

# 2. Replace FA3 attention call with SDPA
code = code.replace(
    "        y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)\n"
    "        y = y.contiguous().view(B, T, -1)",
    "        # PyTorch SDPA (auto-dispatches to FlashAttention 2 on Ada Lovelace)\n"
    "        q = q.transpose(1, 2)  # (B, H, T, D)\n"
    "        k = k.transpose(1, 2)\n"
    "        v = v.transpose(1, 2)\n"
    "        if self.n_kv_head < self.n_head:\n"
    "            r = self.n_head // self.n_kv_head\n"
    "            k = k.repeat_interleave(r, dim=1)\n"
    "            v = v.repeat_interleave(r, dim=1)\n"
    "        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n"
    "        y = y.transpose(1, 2).contiguous().view(B, T, -1)"
)

# 3. Reduce batch size for L4's 24GB VRAM (H100 uses 128 with 80GB)
code = code.replace('DEVICE_BATCH_SIZE = 128', 'DEVICE_BATCH_SIZE = 32')

# 4. Update peak FLOPs for L4 bf16 (~120 TFLOPS vs H100's 989.5)
code = code.replace('H100_BF16_PEAK_FLOPS = 989.5e12', 'L4_BF16_PEAK_FLOPS = 120.0e12')
code = code.replace('H100_BF16_PEAK_FLOPS', 'L4_BF16_PEAK_FLOPS')

with open('train.py', 'w') as f:
    f.write(code)

print('Patched train.py for L4:')
print('  - Flash Attention 3 -> PyTorch SDPA')
print('  - DEVICE_BATCH_SIZE: 128 -> 32')
print('  - Peak FLOPs: 120.0 TFLOPS (L4 bf16)')
print('  - No other changes needed (L4 supports bf16 natively)')

## 6. Restore previous progress

Restores best `train.py` and cumulative `results.tsv` from Google Drive (nightly persistence), then checks the swarm for a better starting point.

In [ ]:
import shutil, os
from coordinator import Coordinator

SAVE_DIR = '/content/drive/MyDrive/autoresearch-swarm'
DRIVE_OK = os.path.exists('/content/drive/MyDrive')

# --- Restore from Google Drive ---
if not DRIVE_OK:
    print('WARNING: Google Drive not mounted. Starting fresh.')
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    saved_train = os.path.join(SAVE_DIR, 'train.py')
    saved_results = os.path.join(SAVE_DIR, 'results.tsv')

    if os.path.exists(saved_train):
        shutil.copy(saved_train, 'train.py')
        print(f'Restored train.py from previous run')

        if os.path.exists(saved_results):
            shutil.copy(saved_results, 'results.tsv')
            best = float('inf')
            n_experiments = 0
            with open('results.tsv') as f:
                for line in f.readlines()[1:]:
                    parts = line.strip().split('\t')
                    n_experiments += 1
                    if len(parts) >= 4 and parts[3].strip() == 'keep':
                        best = min(best, float(parts[1]))
            print(f'Restored results.tsv: {n_experiments} previous experiments, best val_bpb={best:.6f}')
        print('\nContinuing from where you left off!')
    else:
        print('No previous run found in Google Drive. Starting fresh.')
        print(f'Progress will be saved to: {SAVE_DIR}')

# --- Join swarm and check for better config ---
coord = Coordinator()
if not AGENT_CODENAME:
    AGENT_CODENAME = 'colab-l4'  # fallback
coord.agent_id = AGENT_CODENAME

try:
    coord.join_hub(INVITE_TOKEN)
    coord.announce()

    # Check if swarm has a better config for our VRAM tier
    tier_best = coord.pull_best_config_for_tier()
    if tier_best:
        swarm_source, swarm_meta = tier_best
        swarm_bpb = swarm_meta.get('val_bpb', float('inf'))
        swarm_by = swarm_meta.get('agent_id', '?')
        print(f'\nSwarm best for tier "{coord.vram_tier}": val_bpb={swarm_bpb:.6f} (by {swarm_by})')

        # Check if swarm config is better than our local
        local_best = float('inf')
        if os.path.exists('results.tsv'):
            with open('results.tsv') as f:
                for line in f.readlines()[1:]:
                    parts = line.strip().split('\t')
                    if len(parts) >= 4 and parts[3].strip() == 'keep':
                        local_best = min(local_best, float(parts[1]))

        if swarm_bpb < local_best:
            print(f'Adopting swarm config (swarm {swarm_bpb:.6f} < local {local_best:.6f})')
            with open('train.py', 'w') as f:
                f.write(swarm_source)
        else:
            print(f'Local config is better or equal ({local_best:.6f} <= {swarm_bpb:.6f}), keeping it.')
    else:
        print('No swarm configs available yet — you might be the first!')
except Exception as e:
    print(f'Swarm sync failed (non-fatal): {e}')
    print('Continuing with local config.')

## 7. Run training (~5 minutes)

Single training experiment. Watch for `val_bpb` at the end — lower is better.

On first run this is the baseline. On subsequent nights, this confirms the restored model still works.

In [ ]:
!uv run python train.py 2>&1 | tee run.log

## 8. Set up git and record baseline

In [ ]:
import subprocess, datetime, os, shutil

# Git needs an identity on fresh Colab VMs
subprocess.run(['git', 'config', '--global', 'user.email', 'autoresearch@colab'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name', 'autoresearch'], check=True)

# Ignore venv, caches, logs from git
with open('.gitignore', 'w') as f:
    f.write('.venv/\n__pycache__/\n*.pyc\nrun.log\nresults.tsv\nprogress.png\n.autoresearch-key\n')

# Initialize git repo with the current code as baseline
subprocess.run(['git', 'init'], check=True)
subprocess.run(['git', 'add', '-A'], check=True)
subprocess.run(['git', 'commit', '-m', 'L4 + ClimbMix baseline'], check=True)

# Create autoresearch branch with today's date as tag
tag = datetime.date.today().strftime('%b%d').lower()
branch = f'autoresearch/{tag}'
result = subprocess.run(['git', 'checkout', '-b', branch], capture_output=True)
if result.returncode != 0:
    subprocess.run(['git', 'checkout', branch], check=True)

# Initialize results.tsv if not restored from Drive
SAVE_DIR = '/content/drive/MyDrive/autoresearch-swarm'
if not os.path.exists('results.tsv'):
    with open('results.tsv', 'w') as f:
        f.write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')

result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True)
commit_hash = result.stdout.strip()
print(f'Git initialized, on branch: {branch}')
print(f'Baseline commit: {commit_hash}')

In [ ]:
# Record baseline result from run.log
import subprocess, os, shutil

SAVE_DIR = '/content/drive/MyDrive/autoresearch-swarm'

if not os.path.exists('run.log'):
    raise FileNotFoundError("run.log not found — re-run the training cell above first.")

result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True)
commit_hash = result.stdout.strip()

val_bpb = peak_vram_mb = None
with open('run.log') as f:
    for line in f:
        if line.startswith('val_bpb:'):
            val_bpb = float(line.split(':')[1].strip())
        elif line.startswith('peak_vram_mb:'):
            peak_vram_mb = float(line.split(':')[1].strip())

if val_bpb is None:
    print("Training failed. Last 30 lines of run.log:\n")
    with open('run.log') as f:
        lines = f.readlines()
    for line in lines[-30:]:
        print(line, end='')
    raise RuntimeError("Couldn't parse val_bpb — see output above.")

memory_gb = round(peak_vram_mb / 1024, 1)

# Check if this is a continuation
prev_best = float('inf')
if os.path.exists('results.tsv'):
    with open('results.tsv') as f:
        for line in f.readlines()[1:]:
            parts = line.strip().split('\t')
            if len(parts) >= 4 and parts[3].strip() == 'keep':
                prev_best = min(prev_best, float(parts[1]))

if prev_best < float('inf'):
    desc = f'night baseline (restored, val_bpb={val_bpb:.6f})'
    print(f'Continuing from previous best: {prev_best:.6f}')
    print(f'Tonight\'s baseline run: val_bpb={val_bpb:.6f}, memory={memory_gb}GB')
else:
    desc = 'baseline (L4 + ClimbMix)'
    print(f'Recorded baseline: val_bpb={val_bpb:.6f}, memory={memory_gb}GB')

with open('results.tsv', 'a') as f:
    f.write(f'{commit_hash}\t{val_bpb:.6f}\t{memory_gb}\tkeep\t{desc}\n')

# Publish baseline to swarm
try:
    with open('train.py') as f:
        train_source = f.read()
    coord.publish_result(
        experiment_key=coord._make_key(desc),
        val_bpb=val_bpb,
        memory_gb=memory_gb,
        status='keep',
        description=desc,
        train_py_source=train_source,
    )
    print('Published baseline to swarm.')
except Exception as e:
    print(f'Swarm publish failed (non-fatal): {e}')

# Save to Drive
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copy('train.py', os.path.join(SAVE_DIR, 'train.py'))
    shutil.copy('results.tsv', os.path.join(SAVE_DIR, 'results.tsv'))
    print(f'Saved to Google Drive: {SAVE_DIR}')

!cat results.tsv

## 9. Autonomous experiment loop (swarm-connected)

Paste your Anthropic API key below and run the cell. It will iterate autonomously — proposing hypotheses, claiming experiments via the swarm, training, and publishing results.

**Swarm features:**
- Claims experiments before running (prevents duplicates across agents)
- Publishes results + full train.py source to the shared network
- Syncs with swarm's best config every 5 experiments
- Falls back to solo mode if the swarm is unreachable

**Progress is saved to Google Drive after every improvement.**

~12 experiments/hour, ~$3/night with Sonnet.

In [ ]:
# ============================================================
# AUTORESEARCH SWARM — AUTONOMOUS EXPERIMENT LOOP
# ============================================================
# Connected to the autoresearch-at-home swarm via Ensue.
# Runs ~12 experiments/hour (5 min each). Safe to close browser.
# Stop anytime — results are saved after every improvement.
# ============================================================

ANTHROPIC_API_KEY = ""  # <-- PASTE YOUR KEY HERE
MODEL = "claude-sonnet-4-6"       # ~$3/night  (or "claude-opus-4-6" ~$15/night)
MAX_EXPERIMENTS = 200              # safety limit (~17 hours)

assert ANTHROPIC_API_KEY, "Paste your Anthropic API key on the line above, then re-run this cell."

import os, subprocess, re, time, shutil, json
os.system("pip install -q anthropic")
import anthropic
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from coordinator import Coordinator

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SAVE_DIR = '/content/drive/MyDrive/autoresearch-swarm'
INVITE_TOKEN = "43705dda49374a38997f117c87cba9437d715800f1474e17ad170ea7a0ba7316"

# ---- coordinator setup ----

coord = Coordinator()
coord.agent_id = AGENT_CODENAME or 'colab-l4'
try:
    coord.join_hub(INVITE_TOKEN)
    coord.announce()
except Exception as e:
    print(f'Swarm connection failed (will run in solo mode): {e}')

# ---- helpers ----

def get_commit():
    return subprocess.run(
        ['git', 'rev-parse', '--short', 'HEAD'],
        capture_output=True, text=True
    ).stdout.strip()

def run_training():
    """Run train.py, return (val_bpb, peak_vram_mb) or (None, None) on crash."""
    try:
        subprocess.run('uv run python train.py > run.log 2>&1',
                       shell=True, timeout=600)
    except subprocess.TimeoutExpired:
        return None, None
    val_bpb = peak_vram = None
    try:
        with open('run.log') as f:
            for line in f:
                if line.startswith('val_bpb:'):
                    val_bpb = float(line.split(':')[1].strip())
                elif line.startswith('peak_vram_mb:'):
                    peak_vram = float(line.split(':')[1].strip())
    except Exception:
        pass
    return val_bpb, peak_vram

def log_result(commit, val_bpb, memory_gb, status, desc):
    with open('results.tsv', 'a') as f:
        bpb = f"{val_bpb:.6f}" if val_bpb else "0.000000"
        mem = f"{memory_gb:.1f}" if memory_gb else "0.0"
        f.write(f"{commit}\t{bpb}\t{mem}\t{status}\t{desc}\n")

def save_to_drive():
    """Save current best train.py and results.tsv to Google Drive."""
    if not os.path.exists('/content/drive/MyDrive'):
        return
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copy('train.py', os.path.join(SAVE_DIR, 'train.py'))
    shutil.copy('results.tsv', os.path.join(SAVE_DIR, 'results.tsv'))

def plot_progress():
    """Generate progress chart and save to Google Drive."""
    try:
        df = pd.read_csv("results.tsv", sep="\t")
        df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
        df["status"] = df["status"].str.strip().str.upper()

        fig, ax = plt.subplots(figsize=(16, 8))
        valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
        if len(valid) == 0:
            plt.close()
            return
        baseline_bpb = valid.loc[0, "val_bpb"]

        disc = valid[valid["status"] == "DISCARD"]
        ax.scatter(disc.index, disc["val_bpb"],
                   c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

        kept_v = valid[valid["status"] == "KEEP"]
        ax.scatter(kept_v.index, kept_v["val_bpb"],
                   c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

        kept_mask = valid["status"] == "KEEP"
        kept_idx = valid.index[kept_mask]
        kept_bpb = valid.loc[kept_mask, "val_bpb"]
        running_min = kept_bpb.cummin()
        best = running_min.iloc[-1] if len(running_min) > 0 else baseline_bpb
        ax.step(kept_idx, running_min, where="post", color="#27ae60",
                linewidth=2, alpha=0.7, zorder=3, label="Running best")

        for idx, bpb in zip(kept_idx, kept_bpb):
            desc = str(valid.loc[idx, "description"]).strip()
            if len(desc) > 45:
                desc = desc[:42] + "..."
            ax.annotate(desc, (idx, bpb), textcoords="offset points",
                        xytext=(6, 6), fontsize=8.0, color="#1a7a3a", alpha=0.9,
                        rotation=30, ha="left", va="bottom")

        n_total = len(df)
        n_kept = len(df[df["status"] == "KEEP"])
        n_crash = len(df[df["status"] == "CRASH"])
        ax.set_xlabel("Experiment #", fontsize=12)
        ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
        ax.set_title(f"Autoresearch Swarm [{coord.agent_id}]: {n_total} experiments, {n_kept} kept, best={best:.6f}", fontsize=14)
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(True, alpha=0.2)

        all_bpb = valid["val_bpb"].dropna()
        if len(all_bpb) > 1:
            y_min, y_max = all_bpb.min(), all_bpb.max()
            margin = max((y_max - y_min) * 0.1, 0.001)
            ax.set_ylim(y_min - margin, y_max + margin)
        elif len(all_bpb) == 1:
            ax.set_ylim(all_bpb.iloc[0] - 0.005, all_bpb.iloc[0] + 0.005)
        ax.set_xlim(-0.5, max(len(valid) - 0.5, 1.5))

        plt.tight_layout()
        plt.savefig("progress.png", dpi=150, bbox_inches="tight")
        if os.path.exists("/content/drive/MyDrive"):
            shutil.copy("progress.png", os.path.join(SAVE_DIR, "progress.png"))
        clear_output(wait=True)
        plt.show()
    except Exception:
        pass  # don't let plotting errors break the experiment loop

def parse_response(text):
    """Extract description and code from Claude's response."""
    desc_match = re.search(r'DESCRIPTION:\s*(.+)', text)
    desc = desc_match.group(1).strip()[:60] if desc_match else None
    code_match = re.search(r'`{3,}\s*[Pp]ython[^\n]*\n(.*?)`{3,}', text, re.DOTALL)
    if not code_match:
        code_match = re.search(r'`{3,}\n(.*?)`{3,}', text, re.DOTALL)
    code = code_match.group(1) if code_match else None
    return desc, code

def sync_with_swarm(best_bpb):
    """Pull swarm's best config if it beats ours. Returns new best_bpb."""
    try:
        tier_best = coord.pull_best_config_for_tier()
        if tier_best:
            swarm_source, swarm_meta = tier_best
            swarm_bpb = swarm_meta.get('val_bpb', float('inf'))
            if swarm_bpb < best_bpb:
                with open('train.py', 'w') as f:
                    f.write(swarm_source)
                subprocess.run(['git', 'add', 'train.py'], check=True)
                subprocess.run(['git', 'commit', '-m',
                    f'adopt swarm best (val_bpb={swarm_bpb:.6f} from {swarm_meta.get("agent_id", "?")})'],
                    capture_output=True, text=True)
                desc = f'adopt swarm best ({swarm_meta.get("agent_id", "?")}, val_bpb={swarm_bpb:.6f})'
                log_result(get_commit(), swarm_bpb, 0.0, 'keep', desc)
                save_to_drive()
                print(f'Adopted swarm config: val_bpb={swarm_bpb:.6f}')
                return swarm_bpb
            else:
                print(f'Swarm sync: our {best_bpb:.6f} >= swarm {swarm_bpb:.6f}, keeping ours.')
    except Exception as e:
        print(f'Swarm sync failed (non-fatal): {e}')
    return best_bpb

# ---- system prompt ----

SYSTEM_PROMPT = """You are an autonomous ML researcher optimizing a small GPT language model.

GOAL: Achieve the lowest val_bpb (bits per byte) in a fixed 5-minute training budget on an L4 GPU (24GB VRAM, bf16 native, Ada Lovelace architecture).

HARDWARE CONSTRAINTS (critical — read carefully):
- The baseline uses ~11.4GB VRAM with DEVICE_BATCH_SIZE=32, depth=8, ~50M params.
- VRAM must stay under ~20GB. Crashes waste 5 minutes — avoid them.
- Bigger models need more VRAM AND get fewer training steps in 5 minutes.
- Increasing depth from 8 to 12+ or significantly widening the model WILL OOM or undertrain.
- If you want to try a larger model, increase conservatively: e.g. depth 8->9 or 8->10, not 8->12.
- If a "crash" appears in results.tsv, the model was too large. Go smaller.

RULES:
- You can ONLY modify train.py. prepare.py is read-only.
- You cannot add new package imports beyond what's already available.
- Prefer simple changes. A small improvement from a clean change beats a marginal improvement from a hack.
- If removing complexity gives equal results, that's a win — keep it.
- Only try ONE hypothesis per experiment so you can attribute the result.
- Learn from past results: don't repeat failed ideas, build on what worked.

WHAT TO TRY (in this order — exhaust each category before moving on):
1. Learning rate schedule: peak LR, warmup fraction, warmdown fraction, cooldown shape
2. Optimizer tuning: weight decay, Adam betas, Muon momentum, beta2, ns_steps, LR scaling
3. Architecture (same size): activation functions, normalization, attention window sizes, head dim
4. Model size (CAREFUL): small depth/width changes only — e.g. depth 8->9, or adjust ASPECT_RATIO by +/-8
5. Novel ideas: init schemes, residual scaling, embedding tricks

OUTPUT FORMAT (follow EXACTLY):
Line 1: DESCRIPTION: <short description of hypothesis, max 60 chars>
Line 2: (blank)
Lines 3+: The COMPLETE modified train.py inside ```python ... ``` markers.

Output NOTHING else. No explanations, no analysis — just the description line and the full code."""

# ---- read current best from results.tsv ----

best_bpb = float('inf')
with open('results.tsv') as f:
    for line in f.readlines()[1:]:
        parts = line.strip().split('\t')
        if len(parts) >= 4 and parts[3].strip() == 'keep':
            best_bpb = min(best_bpb, float(parts[1]))

assert best_bpb < float('inf'), (
    "No baseline found in results.tsv. "
    "Run the training cell and baseline recording cell first."
)

print("=" * 60)
print("AUTORESEARCH SWARM — AUTONOMOUS LOOP")
print(f"Agent: {coord.agent_id}")
print(f"Model: {MODEL}")
print(f"VRAM tier: {coord.vram_tier}")
print(f"Current best val_bpb: {best_bpb:.6f}")
print(f"Max experiments: {MAX_EXPERIMENTS}")
print(f"Saving to: {SAVE_DIR}")
print("=" * 60)
print()

# ---- main loop ----

for exp in range(MAX_EXPERIMENTS):
    try:
        # Sync with swarm every 5 experiments
        if coord.connected and coord.should_sync():
            best_bpb = sync_with_swarm(best_bpb)

        # Read current state
        with open('train.py') as f:
            current_code = f.read()
        with open('results.tsv') as f:
            current_results = f.read()

        # Ask Claude for next experiment
        response = client.messages.create(
            model=MODEL,
            max_tokens=16384,
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}
            }],
            messages=[{
                "role": "user",
                "content": (
                    f"Current train.py:\n\n```python\n{current_code}\n```\n\n"
                    f"Experiment results so far:\n```\n{current_results}\n```\n\n"
                    f"Propose the next experiment."
                )
            }]
        )

        if response.stop_reason == "max_tokens":
            print(f"[{exp:3d}] SKIP — response truncated (hit max_tokens). Code block incomplete.")
            continue

        text = response.content[0].text
        desc, new_code = parse_response(text)

        if desc is None:
            desc = f"experiment {exp}"

        if new_code is None:
            print(f"[{exp:3d}] SKIP — couldn't parse code. Response preview: {text[:150]}")
            continue

        if 'def forward' not in new_code or 'optimizer' not in new_code:
            print(f"[{exp:3d}] SKIP — response doesn't look like valid train.py")
            continue

        # Claim experiment in swarm (prevents duplicates)
        exp_key = None
        if coord.connected:
            exp_key = coord.claim_experiment(desc)
            if exp_key is None:
                print(f"[{exp:3d}] SKIP — experiment already claimed by another agent: {desc}")
                continue

        # Write new code and commit
        with open('train.py', 'w') as f:
            f.write(new_code)

        subprocess.run(['git', 'add', 'train.py'], check=True)
        commit_result = subprocess.run(
            ['git', 'commit', '-m', desc],
            capture_output=True, text=True
        )
        if commit_result.returncode != 0:
            print(f"[{exp:3d}] SKIP — no changes from model")
            continue

        commit = get_commit()
        print(f"[{exp:3d}] {desc} ({commit})...", end=" ", flush=True)

        # Run training (~5 min)
        t0 = time.time()
        val_bpb, peak_vram = run_training()
        dt = time.time() - t0

        if val_bpb is None:
            print(f"CRASH ({dt:.0f}s)")
            log_result(commit, None, None, "crash", desc)
            subprocess.run(['git', 'reset', '--hard', 'HEAD~1'])
            # Publish crash to swarm
            if coord.connected and exp_key:
                try:
                    coord.publish_result(exp_key, 0.0, 0.0, 'crash', desc, '')
                except Exception:
                    pass
            plot_progress()
            continue

        memory_gb = round(peak_vram / 1024, 1)

        if val_bpb < best_bpb:
            delta = best_bpb - val_bpb
            print(f"KEEP  bpb={val_bpb:.6f} (delta={delta:.6f}) mem={memory_gb}GB ({dt:.0f}s)")
            log_result(commit, val_bpb, memory_gb, "keep", desc)
            best_bpb = val_bpb
            save_to_drive()
            # Publish keep to swarm
            if coord.connected and exp_key:
                try:
                    coord.publish_result(exp_key, val_bpb, memory_gb, 'keep', desc, new_code)
                except Exception:
                    pass
            plot_progress()
        else:
            print(f"DISCARD  bpb={val_bpb:.6f} (best={best_bpb:.6f}) mem={memory_gb}GB ({dt:.0f}s)")
            log_result(commit, val_bpb, memory_gb, "discard", desc)
            subprocess.run(['git', 'reset', '--hard', 'HEAD~1'])
            # Publish discard to swarm
            if coord.connected and exp_key:
                try:
                    coord.publish_result(exp_key, val_bpb, memory_gb, 'discard', desc, new_code)
                except Exception:
                    pass
            plot_progress()

    except anthropic.APIError as e:
        print(f"[{exp:3d}] API ERROR: {e} — waiting 30s...")
        time.sleep(30)
    except Exception as e:
        print(f"[{exp:3d}] ERROR: {e}")
        subprocess.run(['git', 'checkout', '--', 'train.py'], capture_output=True)
        time.sleep(5)

print(f"\n{'=' * 60}")
print(f"DONE — {MAX_EXPERIMENTS} experiments reached")
print(f"Final best val_bpb: {best_bpb:.6f}")
print(f"Run the plotting cell to visualize results!")
print(f"{'=' * 60}")

## 10. Plot progress

Run this cell anytime to see the Karpathy-style progress chart.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_bpb = valid.loc[0, "val_bpb"]

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()
best = running_min.iloc[-1] if len(running_min) > 0 else baseline_bpb
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, bpb), textcoords="offset points",
                xytext=(6, 6), fontsize=8.0, color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
n_crash = len(df[df["status"] == "CRASH"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Swarm Progress: {n_total} Experiments, {n_kept} Kept, {n_crash} Crashed", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

all_bpb = valid["val_bpb"].dropna()
if len(all_bpb) > 1:
    y_min, y_max = all_bpb.min(), all_bpb.max()
    margin = max((y_max - y_min) * 0.1, 0.001)
    ax.set_ylim(y_min - margin, y_max + margin)
elif len(all_bpb) == 1:
    ax.set_ylim(all_bpb.iloc[0] - 0.005, all_bpb.iloc[0] + 0.005)

ax.set_xlim(-0.5, max(len(valid) - 0.5, 1.5))

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBaseline: {baseline_bpb:.6f}  |  Best: {best:.6f}  |  Improvement: {baseline_bpb - best:.6f}")
print(f"Experiments: {n_total}  |  Kept: {n_kept}  |  Crashed: {n_crash}  |  Keep rate: {n_kept}/{n_total}")